# Blog 7 — Schema Evolution in Data Engineering

## Complete, Isolated, Production-Style Databricks Notebook

This notebook creates its own Unity Catalog schema, Delta tables, source datasets, schema versions, validation logic, audit table, and quarantine table.

It does **not** use DBFS root, `/tmp`, or any existing Blog 4/5 objects.

**Catalog:** `workspace`  
**Dedicated schema:** `workspace.blog7_schema_evolution`

### Learning flow

```text
Create Environment
      ↓
Initial Schema
      ↓
Schema Enforcement
      ↓
Additive Evolution
      ↓
Schema Diff
      ↓
Validation
      ↓
Breaking Changes
      ↓
Schema Contract
      ↓
Quarantine + Audit
      ↓
MERGE
      ↓
Incremental Processing
      ↓
Idempotency
      ↓
Production Governance
      ↓
Full Validation
```

# 1. What Is Schema Evolution?

Schema evolution is the controlled handling of changes to the structure of data over time.

Examples:

```text
Add column       → customer_id | name | email | phone
Type change      → age INT → age STRING
Rename           → customer_id → cust_id
Removal          → email disappears
Nested change    → address(city,state) → address(city,state,pincode)
```

The engineering question is:

> Should the pipeline accept the change, evolve the target, review it, or reject it?

# 2. Schema Enforcement vs Schema Evolution

**Schema enforcement** protects the existing target contract.

**Schema evolution** allows an approved change to modify the target schema.

```text
Schema Enforcement = CHECK
Schema Evolution   = CHANGE
```

`mergeSchema` is an implementation mechanism, not a governance policy.

# 3. Medallion Schema Governance

A practical strategy:

| Layer | Strategy |
|---|---|
| Bronze | Preserve source structure; tolerate controlled source changes |
| Silver | Standardize and strongly govern schema |
| Gold | Keep stable business-facing contracts |

The goal is to prevent a source change from automatically breaking every downstream layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import json
from datetime import datetime

CATALOG = "workspace"
SCHEMA = "blog7_schema_evolution"

CUSTOMER_TABLE = f"{CATALOG}.{SCHEMA}.customers"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.schema_change_audit"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.schema_quarantine"

print("Catalog :", CATALOG)
print("Schema  :", SCHEMA)
print("Spark   :", spark.version)
print("Customer:", CUSTOMER_TABLE)

Catalog : workspace
Schema  : blog7_schema_evolution
Spark   : 4.1.0
Customer: workspace.blog7_schema_evolution.customers


# 4. Create the Dedicated Schema

The notebook resets **only** `workspace.blog7_schema_evolution`.

This makes the project rerunnable.

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

spark.sql(f"""
CREATE SCHEMA {CATALOG}.{SCHEMA}
COMMENT 'Isolated Blog 7 - Schema Evolution'
""")

print(f"Created {CATALOG}.{SCHEMA}")

Created workspace.blog7_schema_evolution


In [0]:
spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{SCHEMA}").show(truncate=False)

+-------------------------+---------------------------------------------------------+
|database_description_item|database_description_value                               |
+-------------------------+---------------------------------------------------------+
|Catalog Name             |workspace                                                |
|Namespace Name           |blog7_schema_evolution                                   |
|Comment                  |Isolated Blog 7 - Schema Evolution                       |
|Location                 |                                                         |
|Owner                    |bharath2704.a@gmail.com                                  |
|Properties               |                                                         |
|Predictive Optimization  |ENABLE (inherited from METASTORE metastore_aws_us_east_2)|
+-------------------------+---------------------------------------------------------+



# 5. Create Version 1 — Initial Customer Data

Initial contract:

```text
customer_id INT
name        STRING
email       STRING
city        STRING
```

The schema is explicitly defined so the demonstration is deterministic.

In [0]:
schema_v1 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True)
])

data_v1 = [
    (1, "Alice", "alice@example.com", "Chennai"),
    (2, "Bob", "bob@example.com", "Coimbatore"),
    (3, "Charlie", "charlie@example.com", "Madurai")
]

df_v1 = spark.createDataFrame(data_v1, schema_v1)

df_v1.printSchema()
display(df_v1)

root
 |-- customer_id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)



customer_id,name,email,city
1,Alice,alice@example.com,Chennai
2,Bob,bob@example.com,Coimbatore
3,Charlie,charlie@example.com,Madurai


In [0]:
df_v1.write     .format("delta")     .mode("overwrite")     .saveAsTable(CUSTOMER_TABLE)

print("Created managed Delta table:", CUSTOMER_TABLE)
spark.table(CUSTOMER_TABLE).printSchema()
display(spark.table(CUSTOMER_TABLE))

Created managed Delta table: workspace.blog7_schema_evolution.customers
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)



customer_id,name,email,city
1,Alice,alice@example.com,Chennai
2,Bob,bob@example.com,Coimbatore
3,Charlie,charlie@example.com,Madurai


# 6. Create Audit and Quarantine Tables

Production schema evolution should be observable.

**Audit** records schema decisions.

**Quarantine** records rejected schema-change events.

In [0]:
audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("classification", StringType(), True),
    StructField("decision", StringType(), True),
    StructField("added_columns", StringType(), True),
    StructField("removed_columns", StringType(), True),
    StructField("type_changes", StringType(), True),
    StructField("nullability_changes", StringType(), True),
    StructField("reason", StringType(), True)
])

quarantine_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("rejection_reason", StringType(), True),
    StructField("schema_details", StringType(), True)
])

spark.createDataFrame([], audit_schema)     .write.format("delta").mode("overwrite").saveAsTable(AUDIT_TABLE)

spark.createDataFrame([], quarantine_schema)     .write.format("delta").mode("overwrite").saveAsTable(QUARANTINE_TABLE)

print("Audit table:", AUDIT_TABLE)
print("Quarantine table:", QUARANTINE_TABLE)

Audit table: workspace.blog7_schema_evolution.schema_change_audit
Quarantine table: workspace.blog7_schema_evolution.schema_quarantine


# 7. Capture the Target Schema

The current target schema is the baseline for every incoming-batch comparison.

In [0]:
target_schema = spark.table(CUSTOMER_TABLE).schema

for field in target_schema.fields:
    print(
        f"{field.name:15} "
        f"type={field.dataType.simpleString():10} "
        f"nullable={field.nullable}"
    )

customer_id     type=int        nullable=True
name            type=string     nullable=True
email           type=string     nullable=True
city            type=string     nullable=True


# 8. Build the Schema-Diff Engine

We separate:

### Structural schema
- column names
- data types

from:

### Contract metadata
- nullability
- business meaning
- business rules

This is important because Spark may expose nullable metadata differently after a Delta write/read cycle.

The primary schema-evolution checks are:

```text
Column added?
Column removed?
Data type changed?
```

Nullability is evaluated separately.

In [0]:
def structural_schema_diff(expected_schema, incoming_schema):
    expected = {
        field.name: field.dataType.simpleString()
        for field in expected_schema.fields
    }
    incoming = {
        field.name: field.dataType.simpleString()
        for field in incoming_schema.fields
    }

    expected_columns = set(expected)
    incoming_columns = set(incoming)

    added_columns = sorted(incoming_columns - expected_columns)
    removed_columns = sorted(expected_columns - incoming_columns)

    type_changes = []

    for column in sorted(expected_columns & incoming_columns):
        if expected[column] != incoming[column]:
            type_changes.append({
                "column": column,
                "expected_type": expected[column],
                "incoming_type": incoming[column]
            })

    return {
        "added_columns": added_columns,
        "removed_columns": removed_columns,
        "type_changes": type_changes
    }


def nullability_diff(expected_schema, incoming_schema):
    expected = {field.name: field.nullable for field in expected_schema.fields}
    incoming = {field.name: field.nullable for field in incoming_schema.fields}

    changes = []

    for column in sorted(set(expected) & set(incoming)):
        if expected[column] != incoming[column]:
            changes.append({
                "column": column,
                "expected_nullable": expected[column],
                "incoming_nullable": incoming[column]
            })

    return changes

# 9. Schema Change Classification

For this project:

| Change | Classification |
|---|---|
| No structural change | `NO_CHANGE` |
| New column | `EVOLVE` |
| Data-type change | `REJECT` |
| Column removal | `REJECT` |
| Rename | `REJECT` / explicit migration |
| Nullability-only difference | `REVIEW` |

A new column is classified as `EVOLVE` even if Spark exposes an unrelated nullable-metadata difference on existing fields. That keeps structural evolution separate from contract review.

In [0]:
def classify_schema_change(expected_schema, incoming_schema):
    structural = structural_schema_diff(expected_schema, incoming_schema)
    nullable = nullability_diff(expected_schema, incoming_schema)

    if structural["removed_columns"]:
        classification = "REJECT"
    elif structural["type_changes"]:
        classification = "REJECT"
    elif structural["added_columns"]:
        classification = "EVOLVE"
    elif nullable:
        classification = "REVIEW"
    else:
        classification = "NO_CHANGE"

    return {
        "classification": classification,
        "structural_diff": structural,
        "nullability_changes": nullable
    }

# 10. Version 2 — Additive Schema Evolution

The source adds:

```text
phone STRING
```

Expected:

```text
added_columns = ['phone']
classification = EVOLVE
```

In [0]:
schema_v2 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("phone", StringType(), True)
])

data_v2 = [
    (1, "Alice", "alice@example.com", "Chennai", "9876500001"),
    (2, "Bob", "bob@example.com", "Coimbatore", "9876500002"),
    (3, "Charlie", "charlie@example.com", "Madurai", "9876500003")
]

df_v2 = spark.createDataFrame(data_v2, schema_v2)

decision_v2 = classify_schema_change(
    spark.table(CUSTOMER_TABLE).schema,
    df_v2.schema
)

print(json.dumps(decision_v2, indent=2))

assert decision_v2["structural_diff"]["added_columns"] == ["phone"]
assert decision_v2["structural_diff"]["removed_columns"] == []
assert decision_v2["structural_diff"]["type_changes"] == []
assert decision_v2["classification"] == "EVOLVE"

print("Additive schema validation passed.")

{
  "classification": "EVOLVE",
  "structural_diff": {
    "added_columns": [
      "phone"
    ],
    "removed_columns": [],
    "type_changes": []
  },
  "nullability_changes": [
    {
      "column": "customer_id",
      "expected_nullable": true,
      "incoming_nullable": false
    }
  ]
}
Additive schema validation passed.


# 11. Demonstrate Schema Enforcement

Try the new schema without `mergeSchema`.

Expected:

```text
Write rejected
```

In [0]:
try:
    df_v2.write         .format("delta")         .mode("append")         .saveAsTable(CUSTOMER_TABLE)

    enforcement_blocked = False

except Exception as e:
    enforcement_blocked = True
    print("Expected schema enforcement failure:")
    print(str(e)[:2000])

assert enforcement_blocked
print("Schema enforcement validation passed.")

Expected schema enforcement failure:
[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 95b7274e-3619-49c0-b161-ae2d175fb71b).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)


Data schema:
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: string (nullable = true)


- Table ACLs are enabled in this cluster, so automatic schema migration is not allowed.

# 12. Apply Approved Schema Evolution

The governance layer has already classified the change as:

```text
EVOLVE
```

Only now do we use:

```text
mergeSchema = true
```

In [0]:
if decision_v2["classification"] != "EVOLVE":
    raise ValueError("Schema evolution was not approved.")

df_v2.write     .format("delta")     .mode("append")     .option("mergeSchema", "true")     .saveAsTable(CUSTOMER_TABLE)

evolved_v2 = spark.table(CUSTOMER_TABLE)

assert "phone" in evolved_v2.columns

print("Schema evolution succeeded.")
evolved_v2.printSchema()

Schema evolution succeeded.
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: string (nullable = true)



In [0]:
expected_columns_v2 = {
    "customer_id",
    "name",
    "email",
    "city",
    "phone"
}

assert expected_columns_v2.issubset(set(evolved_v2.columns))
assert evolved_v2.count() == 6

print("Post-evolution validation passed.")
display(evolved_v2.orderBy("customer_id"))

Post-evolution validation passed.


customer_id,name,email,city,phone
1,Alice,alice@example.com,Chennai,null
1,Alice,alice@example.com,Chennai,9876500001
2,Bob,bob@example.com,Coimbatore,9876500002
2,Bob,bob@example.com,Coimbatore,null
3,Charlie,charlie@example.com,Madurai,9876500003
3,Charlie,charlie@example.com,Madurai,null


# 13. Important Observation

We started with:

```text
3 rows
4 columns
```

Then appended Version 2:

```text
3 rows
5 columns
```

The table now contains:

```text
6 rows
5 columns
```

This is intentionally an **append demonstration**.

In production, schema evolution and record-level processing are separate decisions.

# 14. Version 3 — Schema Evolution + Delta MERGE

Now combine:

```text
Schema Evolution
+
Incremental Data
+
Delta MERGE
```

Version 3 adds:

```text
country STRING
```

and contains:

- an update for Alice
- an existing Bob record
- a new David record

In [0]:
schema_v3 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("country", StringType(), True)
])

data_v3 = [
    (1, "Alice", "alice@newdomain.com", "Chennai", "9876500001", "India"),
    (2, "Bob", "bob@example.com", "Coimbatore", "9876500002", "India"),
    (4, "David", "david@example.com", "Salem", "9876500004", "India")
]

df_v3 = spark.createDataFrame(data_v3, schema_v3)

decision_v3 = classify_schema_change(
    spark.table(CUSTOMER_TABLE).schema,
    df_v3.schema
)

print(json.dumps(decision_v3, indent=2))

assert decision_v3["classification"] == "EVOLVE"
assert decision_v3["structural_diff"]["added_columns"] == ["country"]

{
  "classification": "EVOLVE",
  "structural_diff": {
    "added_columns": [
      "country"
    ],
    "removed_columns": [],
    "type_changes": []
  },
  "nullability_changes": [
    {
      "column": "customer_id",
      "expected_nullable": true,
      "incoming_nullable": false
    }
  ]
}


# 15. Evolve the Table Schema Without Inserting Rows

We first evolve only the schema.

A zero-row DataFrame introduces the new column without inserting the Version 3 records.

In [0]:
(
    df_v3
    .select("country")
    .limit(0)
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(CUSTOMER_TABLE)
)

assert "country" in spark.table(CUSTOMER_TABLE).columns
print("Schema after country evolution:")
spark.table(CUSTOMER_TABLE).printSchema()

Schema after country evolution:
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country: string (nullable = true)



# 16. Run the Delta MERGE

`customer_id` is the business key.

Existing customers are updated.

New customers are inserted.

In [0]:
target = DeltaTable.forName(spark, CUSTOMER_TABLE)

(
    target.alias("t")
    .merge(
        df_v3.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

result = spark.table(CUSTOMER_TABLE)

display(
    result
    .select(
        "customer_id",
        "name",
        "email",
        "city",
        "phone",
        "country"
    )
    .orderBy("customer_id")
)

customer_id,name,email,city,phone,country
1,Alice,alice@newdomain.com,Chennai,9876500001,India
1,Alice,alice@newdomain.com,Chennai,9876500001,India
2,Bob,bob@example.com,Coimbatore,9876500002,India
2,Bob,bob@example.com,Coimbatore,9876500002,India
3,Charlie,charlie@example.com,Madurai,null,null
3,Charlie,charlie@example.com,Madurai,9876500003,null
4,David,david@example.com,Salem,9876500004,India


In [0]:
alice_email = (
    result
    .filter(F.col("customer_id") == 1)
    .select("email")
    .collect()
)

david_count = (
    result
    .filter(F.col("customer_id") == 4)
    .count()
)

assert "country" in result.columns
assert len(alice_email) >= 1
assert all(record["email"] == "alice@newdomain.com" for record in alice_email)
assert david_count == 1

print("MERGE + schema evolution validation passed.")

MERGE + schema evolution validation passed.


# 17. Breaking Change — Data-Type Change

Example:

```text
customer_id INT
      ↓
customer_id STRING
```

Even if values such as `"1"` look convertible, the contract changed.

Potential impacts:

- joins
- filters
- aggregations
- downstream contracts
- BI tools
- applications

Default policy:

```text
REJECT
```

In [0]:
schema_type_change = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("country", StringType(), True)
])

df_type_change = spark.createDataFrame(
    [
        (
            "1",
            "Alice",
            "alice@newdomain.com",
            "Chennai",
            "9876500001",
            "India"
        )
    ],
    schema_type_change
)

type_decision = classify_schema_change(
    result.schema,
    df_type_change.schema
)

print(json.dumps(type_decision, indent=2))

assert type_decision["classification"] == "REJECT"
assert any(
    change["column"] == "customer_id"
    for change in type_decision["structural_diff"]["type_changes"]
)

print("Type-change rejection validation passed.")

{
  "classification": "REJECT",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [],
    "type_changes": [
      {
        "column": "customer_id",
        "expected_type": "int",
        "incoming_type": "string"
      }
    ]
  },
  "nullability_changes": []
}
Type-change rejection validation passed.


# 18. Breaking Change — Column Rename

Example:

```text
customer_id
     ↓
cust_id
```

A generic diff sees:

```text
Added   → cust_id
Removed → customer_id
```

It should not guess that these are the same business field.

A rename should be an explicit migration.

In [0]:
df_rename = result.select(
    F.col("customer_id").alias("cust_id"),
    "name",
    "email",
    "city",
    "phone",
    "country"
)

rename_decision = classify_schema_change(
    result.schema,
    df_rename.schema
)

print(json.dumps(rename_decision, indent=2))

assert rename_decision["classification"] == "REJECT"
assert "cust_id" in rename_decision["structural_diff"]["added_columns"]
assert "customer_id" in rename_decision["structural_diff"]["removed_columns"]

print("Rename rejection validation passed.")

{
  "classification": "REJECT",
  "structural_diff": {
    "added_columns": [
      "cust_id"
    ],
    "removed_columns": [
      "customer_id"
    ],
    "type_changes": []
  },
  "nullability_changes": []
}
Rename rejection validation passed.


# 19. Breaking Change — Column Removal

If the source stops sending `email`, downstream consumers may fail.

Default:

```text
Removal
   ↓
REJECT
   ↓
Impact Analysis
   ↓
Explicit Migration
```

In [0]:
df_removed = result.select(
    "customer_id",
    "name",
    "city",
    "phone",
    "country"
)

removal_decision = classify_schema_change(
    result.schema,
    df_removed.schema
)

print(json.dumps(removal_decision, indent=2))

assert removal_decision["classification"] == "REJECT"
assert "email" in removal_decision["structural_diff"]["removed_columns"]

print("Removal rejection validation passed.")

{
  "classification": "REJECT",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [
      "email"
    ],
    "type_changes": []
  },
  "nullability_changes": []
}
Removal rejection validation passed.


# 20. Nullability — Separate Contract Review

Nullability is important contract metadata.

Example:

```text
email NULL
   ↓
email NOT NULL
```

We evaluate it separately from structural schema evolution.

If nullability is the only difference:

```text
Classification → REVIEW
```

This avoids Spark's nullable metadata interfering with the main additive-evolution lesson.

In [0]:
nullable_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), False),
    StructField("city", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("country", StringType(), True)
])

df_nullable_change = spark.createDataFrame(
    [
        (
            1,
            "Alice",
            "alice@newdomain.com",
            "Chennai",
            "9876500001",
            "India"
        )
    ],
    nullable_schema
)

nullable_decision = classify_schema_change(
    result.schema,
    df_nullable_change.schema
)

print(json.dumps(nullable_decision, indent=2))
print("Nullability changes:", nullable_decision["nullability_changes"])

{
  "classification": "REVIEW",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [],
    "type_changes": []
  },
  "nullability_changes": [
    {
      "column": "customer_id",
      "expected_nullable": true,
      "incoming_nullable": false
    },
    {
      "column": "email",
      "expected_nullable": true,
      "incoming_nullable": false
    }
  ]
}
Nullability changes: [{'column': 'customer_id', 'expected_nullable': True, 'incoming_nullable': False}, {'column': 'email', 'expected_nullable': True, 'incoming_nullable': False}]


# 21. Nested Schema Evolution

Semi-structured data can evolve inside nested structures.

Version 1:

```text
address:
    city
    state
```

Version 2:

```text
address:
    city
    state
    pincode
```

The governance principle remains:

```text
Detect
 ↓
Classify
 ↓
Validate
 ↓
Approve
 ↓
Evolve
```

Exact automatic nested-evolution behavior should be tested against the actual production Delta/Databricks runtime and write operation.

In [0]:
nested_v1 = spark.createDataFrame(
    [(1, ("Chennai", "Tamil Nadu"))],
    "customer_id INT, address STRUCT<city:STRING,state:STRING>"
)

nested_v2 = spark.createDataFrame(
    [(1, ("Chennai", "Tamil Nadu", "600001"))],
    "customer_id INT, address STRUCT<city:STRING,state:STRING,pincode:STRING>"
)

print("Nested Version 1:")
nested_v1.printSchema()

print("Nested Version 2:")
nested_v2.printSchema()

Nested Version 1:
root
 |-- customer_id: integer (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)

Nested Version 2:
root
 |-- customer_id: integer (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- pincode: string (nullable = true)



# 22. Explicit Silver Schema Contract

A governed Silver layer should have an explicit contract.

For this project:

```text
customer_id → int
name        → string
email       → string
city        → string
phone       → string
country     → string
```

The contract can live in version-controlled configuration, metadata, or a schema registry.

In [0]:
EXPECTED_CONTRACT = {
    "customer_id": "int",
    "name": "string",
    "email": "string",
    "city": "string",
    "phone": "string",
    "country": "string"
}

def contract_diff(df, contract):
    incoming = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    expected = dict(contract)

    return {
        "missing": sorted(set(expected) - set(incoming)),
        "unexpected": sorted(set(incoming) - set(expected)),
        "type_mismatch": [
            {
                "column": column,
                "expected": expected[column],
                "incoming": incoming[column]
            }
            for column in sorted(set(expected) & set(incoming))
            if expected[column] != incoming[column]
        ]
    }


def validate_contract(df, contract):
    details = contract_diff(df, contract)

    valid = (
        not details["missing"]
        and not details["unexpected"]
        and not details["type_mismatch"]
    )

    return valid, details

In [0]:
valid, contract_details = validate_contract(
    result,
    EXPECTED_CONTRACT
)

print("Contract valid:", valid)
print("Contract details:", contract_details)

assert valid
print("Schema contract validation passed.")

Contract valid: True
Contract details: {'missing': [], 'unexpected': [], 'type_mismatch': []}
Schema contract validation passed.


# 23. Reject an Invalid Batch Before Silver

Trusted-layer pattern:

```text
Incoming Batch
      ↓
Schema Contract
      ↓
Schema Classification
      ↓
Data Quality
      ↓
Deduplication
      ↓
MERGE
      ↓
Silver
```

A breaking schema should not modify the trusted target accidentally.

In [0]:
bad_valid, bad_details = validate_contract(
    df_type_change,
    EXPECTED_CONTRACT
)

print("Valid:", bad_valid)
print("Details:", bad_details)

assert not bad_valid
print("Invalid contract correctly rejected.")

Valid: False
Details: {'missing': [], 'unexpected': [], 'type_mismatch': [{'column': 'customer_id', 'expected': 'int', 'incoming': 'string'}]}
Invalid contract correctly rejected.


# 24. Quarantine a Rejected Batch

Production pattern:

```text
Bad Batch
   ↓
Reject
   ↓
Quarantine
   ↓
Audit
   ↓
Alert / Investigate
```

In [0]:
batch_id = "blog7-type-change-001"
event_time = datetime.utcnow()

quarantine_row = [(
    batch_id,
    event_time,
    "customer_source",
    CUSTOMER_TABLE,
    "Incompatible customer_id data type",
    json.dumps(type_decision)
)]

spark.createDataFrame(
    quarantine_row,
    quarantine_schema
).write  .format("delta")  .mode("append")  .saveAsTable(QUARANTINE_TABLE)

display(spark.table(QUARANTINE_TABLE))

/home/spark-cf187b78-bf10-4003-9fb0-54/.ipykernel/82/command-5249718269716426-316624458:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  event_time = datetime.utcnow()


batch_id,event_timestamp,source_system,target_table,rejection_reason,schema_details
blog7-type-change-001,2026-08-19T11:14:52.941Z,customer_source,workspace.blog7_schema_evolution.customers,Incompatible customer_id data type,"{""classification"": ""REJECT"", ""structural_diff"": {""added_columns"": [], ""removed_columns"": [], ""type_changes"": [{""column"": ""customer_id"", ""expected_type"": ""int"", ""incoming_type"": ""string""}]}, ""nullability_changes"": []}"


# 25. Audit the Rejected Schema Change

Audit should tell us:

- batch ID
- timestamp
- source
- target
- classification
- decision
- added columns
- removed columns
- type changes
- nullability changes
- reason

In [0]:
audit_row = [(
    batch_id,
    event_time,
    "customer_source",
    CUSTOMER_TABLE,
    type_decision["classification"],
    "QUARANTINE",
    json.dumps(type_decision["structural_diff"]["added_columns"]),
    json.dumps(type_decision["structural_diff"]["removed_columns"]),
    json.dumps(type_decision["structural_diff"]["type_changes"]),
    json.dumps(type_decision["nullability_changes"]),
    "Incompatible data-type change"
)]

spark.createDataFrame(
    audit_row,
    audit_schema
).write  .format("delta")  .mode("append")  .saveAsTable(AUDIT_TABLE)

display(spark.table(AUDIT_TABLE))

batch_id,event_timestamp,source_system,target_table,classification,decision,added_columns,removed_columns,type_changes,nullability_changes,reason
blog7-type-change-001,2026-08-19T11:14:52.941Z,customer_source,workspace.blog7_schema_evolution.customers,REJECT,QUARANTINE,[],[],"[{""column"": ""customer_id"", ""expected_type"": ""int"", ""incoming_type"": ""string""}]",[],Incompatible data-type change


# 26. Incremental Processing + Schema Evolution

Production sequence:

```text
Source
  ↓
Incremental Read
  ↓
Schema Diff
  ↓
Classification
  ↓
Schema Contract
  ↓
Data Quality
  ↓
Deduplication
  ↓
MERGE
  ↓
Watermark / State Update
```

Key principle:

> **Validate the incoming batch before modifying the trusted target.**

# 27. Idempotency + Schema Evolution

Consider:

```text
Batch 101
   ↓
Schema accepted
   ↓
MERGE succeeds
   ↓
State update fails
   ↓
Retry Batch 101
```

The schema decision should remain deterministic.

A retry should not randomly switch between ACCEPT and REJECT unless the schema contract intentionally changed.

This connects Blog 7 directly with Blog 6's idempotency concepts.

# 28. Schema Evolution + Data Quality

Schema validation answers:

> Is the structure acceptable?

Data quality answers:

> Is the data itself acceptable?

Both are required.

For example:

```text
phone STRING
```

can still contain:

```text
"abc"
```

So:

```text
Schema Valid
      ≠
Data Valid
```

# 29. Production Governance Policy

| Change | Default |
|---|---|
| New nullable column | Approve after validation |
| New nested field | Review |
| Data-type change | Reject |
| Column rename | Reject / explicit migration |
| Column removal | Reject |
| Nullability change | Review |
| Semantic meaning change | Reject / business review |

The policy should be explicit rather than hidden inside a write option.

# 30. Complete Schema Decision Engine

```text
Incoming Schema
      ↓
Structural Diff
      ↓
Nullability Diff
      ↓
Classification
      ↓
Contract Validation
      ↓
Policy
      ↓
┌───────────────┬─────────────────┐
↓               ↓
Approved       Rejected / Review
↓               ↓
Evolve         Quarantine
↓               ↓
Process         Audit
↓
Verify
```

In [0]:
def schema_decision(expected_schema, incoming_schema):
    result = classify_schema_change(
        expected_schema,
        incoming_schema
    )

    decision_map = {
        "NO_CHANGE": "ACCEPT",
        "EVOLVE": "ACCEPT_WITH_APPROVAL",
        "REVIEW": "MANUAL_REVIEW",
        "REJECT": "REJECT"
    }

    return {
        "classification": result["classification"],
        "decision": decision_map[result["classification"]],
        "structural_diff": result["structural_diff"],
        "nullability_changes": result["nullability_changes"]
    }


print("NO CHANGE")
print(json.dumps(schema_decision(result.schema, result.schema), indent=2))

print("ADDITIVE")
print(json.dumps(schema_decision(result.schema, df_v2.schema), indent=2))

print("TYPE CHANGE")
print(json.dumps(schema_decision(result.schema, df_type_change.schema), indent=2))

print("RENAME")
print(json.dumps(schema_decision(result.schema, df_rename.schema), indent=2))

print("REMOVAL")
print(json.dumps(schema_decision(result.schema, df_removed.schema), indent=2))

NO CHANGE
{
  "classification": "NO_CHANGE",
  "decision": "ACCEPT",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [],
    "type_changes": []
  },
  "nullability_changes": []
}
ADDITIVE
{
  "classification": "REJECT",
  "decision": "REJECT",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [
      "country"
    ],
    "type_changes": []
  },
  "nullability_changes": [
    {
      "column": "customer_id",
      "expected_nullable": true,
      "incoming_nullable": false
    }
  ]
}
TYPE CHANGE
{
  "classification": "REJECT",
  "decision": "REJECT",
  "structural_diff": {
    "added_columns": [],
    "removed_columns": [],
    "type_changes": [
      {
        "column": "customer_id",
        "expected_type": "int",
        "incoming_type": "string"
      }
    ]
  },
  "nullability_changes": []
}
RENAME
{
  "classification": "REJECT",
  "decision": "REJECT",
  "structural_diff": {
    "added_columns": [
      "cust_id"
    ],
    "removed_c

# 31. Full Validation Suite

Expected:

```text
No change        → NO_CHANGE
New column       → EVOLVE
Type change      → REJECT
Rename           → REJECT
Column removal   → REJECT
Valid contract   → PASS
Invalid contract → FAIL / reject
```

In [0]:
no_change = classify_schema_change(
    result.schema,
    result.schema
)

assert no_change["classification"] == "NO_CHANGE"
assert decision_v2["classification"] == "EVOLVE"
assert type_decision["classification"] == "REJECT"
assert rename_decision["classification"] == "REJECT"
assert removal_decision["classification"] == "REJECT"

assert valid is True
assert bad_valid is False

final_table = spark.table(CUSTOMER_TABLE)

assert "phone" in final_table.columns
assert "country" in final_table.columns
assert final_table.filter(F.col("customer_id") == 4).count() == 1

assert spark.table(AUDIT_TABLE).count() >= 1
assert spark.table(QUARANTINE_TABLE).count() >= 1

print("==============================================")
print("ALL BLOG 7 VALIDATION TESTS PASSED")
print("==============================================")

ALL BLOG 7 VALIDATION TESTS PASSED


# 32. Inspect the Final Project

### Tables

```text
workspace.blog7_schema_evolution.customers
workspace.blog7_schema_evolution.schema_change_audit
workspace.blog7_schema_evolution.schema_quarantine
```

### Final customer schema

```text
customer_id
name
email
city
phone
country
```

In [0]:
print("TABLES")
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

print("FINAL CUSTOMER SCHEMA")
spark.table(CUSTOMER_TABLE).printSchema()

print("FINAL CUSTOMER DATA")
display(
    spark.table(CUSTOMER_TABLE).orderBy("customer_id")
)

print("AUDIT")
display(spark.table(AUDIT_TABLE))

print("QUARANTINE")
display(spark.table(QUARANTINE_TABLE))

TABLES
+----------------------+-------------------+-----------+
|database              |tableName          |isTemporary|
+----------------------+-------------------+-----------+
|blog7_schema_evolution|customers          |false      |
|blog7_schema_evolution|schema_change_audit|false      |
|blog7_schema_evolution|schema_quarantine  |false      |
+----------------------+-------------------+-----------+

FINAL CUSTOMER SCHEMA
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country: string (nullable = true)

FINAL CUSTOMER DATA


customer_id,name,email,city,phone,country
1,Alice,alice@newdomain.com,Chennai,9876500001,India
1,Alice,alice@newdomain.com,Chennai,9876500001,India
2,Bob,bob@example.com,Coimbatore,9876500002,India
2,Bob,bob@example.com,Coimbatore,9876500002,India
3,Charlie,charlie@example.com,Madurai,9876500003,null
3,Charlie,charlie@example.com,Madurai,null,null
4,David,david@example.com,Salem,9876500004,India


AUDIT


batch_id,event_timestamp,source_system,target_table,classification,decision,added_columns,removed_columns,type_changes,nullability_changes,reason
blog7-type-change-001,2026-08-19T11:14:52.941Z,customer_source,workspace.blog7_schema_evolution.customers,REJECT,QUARANTINE,[],[],"[{""column"": ""customer_id"", ""expected_type"": ""int"", ""incoming_type"": ""string""}]",[],Incompatible data-type change


QUARANTINE


batch_id,event_timestamp,source_system,target_table,rejection_reason,schema_details
blog7-type-change-001,2026-08-19T11:14:52.941Z,customer_source,workspace.blog7_schema_evolution.customers,Incompatible customer_id data type,"{""classification"": ""REJECT"", ""structural_diff"": {""added_columns"": [], ""removed_columns"": [], ""type_changes"": [{""column"": ""customer_id"", ""expected_type"": ""int"", ""incoming_type"": ""string""}]}, ""nullability_changes"": []}"


# 33. Production Validation Checklist

## Environment

- [x] Dedicated Unity Catalog schema
- [x] No DBFS root
- [x] No `/tmp`
- [x] No dependency on previous blogs
- [x] Rerunnable setup

## Schema Evolution

- [x] Initial schema
- [x] Schema enforcement
- [x] Additive evolution
- [x] `mergeSchema`
- [x] Schema diff
- [x] Type-change detection
- [x] Rename detection
- [x] Removal detection
- [x] Nullability review
- [x] Nested schema discussion

## Production Engineering

- [x] Explicit schema contract
- [x] Pre-trusted-layer validation
- [x] Delta MERGE integration
- [x] Quarantine
- [x] Audit
- [x] Incremental-processing relationship
- [x] Idempotency relationship
- [x] Data-quality relationship
- [x] Governance policy
- [x] Full validation suite

# 34. Final Mental Model

Schema evolution is **not**:

```text
mergeSchema = true
```

It is:

```text
                 INCOMING DATA
                       ↓
                 Inspect Schema
                       ↓
                Structural Diff
                       ↓
               Contract Metadata
                       ↓
                  Classify
                       ↓
             ┌─────────┴─────────┐
             ↓                   ↓
       Compatible             Breaking
             ↓                   ↓
          Validate             Reject
             ↓                   ↓
          Approve           Quarantine
             ↓                   ↓
          Evolve                Audit
             ↓
          Process
             ↓
          Verify
```

### Core principle

> **Detect → Classify → Validate → Approve → Evolve — or Reject, Quarantine and Audit.**

# 35. Blog 7 Complete

The concepts now connect together:

```text
Source
  ↓
Bronze
  ↓
Schema Governance
  ↓
Data Quality
  ↓
Silver
  ↓
SCD / MERGE
  ↓
Incremental Processing
  ↓
Idempotency
  ↓
Gold
```

## Next Blog

# Blog 8 — Delta Lake Maintenance

Topics:

```text
OPTIMIZE
VACUUM
Z-Ordering
Small-file problem
File layout
Data skipping
Maintenance strategy
Production scheduling
```